In [ ]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
from pathlib import Path
from scipy.stats import kruskal
import matplotlib.pyplot as plt
from collections import defaultdict


def merged_lesions_csv(dir_path: str | Path) -> dict[str, pd.DataFrame]:
    """
    Scans a directory for spine lesion radiomics CSV files, groups them by their
    imaging type, and merges them into centralized DataFrames per image modality.
    """
    base_dir = Path(dir_path)

    # 1. Use pathlib's rglob for deep scanning of the target CSV pattern
    all_csv_files = base_dir.rglob("*spine_lesions*.csv")

    # Registry to group file paths by their filename: {csv_name: [Path, Path, ...]}
    csv_groups = defaultdict(list)
    result_dict = {}

    # 2. Group discovered CSV files by their exact filename
    for file_path in all_csv_files:
        csv_groups[file_path.name].append(file_path)

    # 3. Process each group and merge individual patient tables
    for csv_name, file_paths in csv_groups.items():
        loaded_dfs = []

        for file_path in file_paths:
            # Read the target CSV file
            df = pd.read_csv(file_path)

            # Extract the patient folder name (parent directory of the file)
            patient_name = file_path.parent.name

            # Insert the patient identifier safely as the very first column
            df.insert(0, "patient", patient_name)
            loaded_dfs.append(df)

        if not loaded_dfs:
            continue

        # Clean the key name by removing the standard suffix to keep it short
        clean_key_name = csv_name.replace("_radiomics_spine_lesions_features.csv", "")

        # Concatenate all patient rows into one large master DataFrame for this modality
        master_df = pd.concat(loaded_dfs, ignore_index=True)
        result_dict[clean_key_name] = master_df

    return result_dict

In [ ]:
# 1. Base configuration and directory definitions
MAIN_FOLDER = Path(r"E:\DATA_Myelomy")
TARGET_FILE_NAME = "BMD_radiomics_vertebrae_features.csv"

all_patient_records = []

# ==================================================================
# 2. DATA EXTRACTION: Scanning Patient Folders for Vertebrae Metrics
# ==================================================================
if MAIN_FOLDER.exists():
    for patient_folder in MAIN_FOLDER.iterdir():
        # Process only actual directories
        if not patient_folder.is_dir():
            continue

        # Target specific vertebrae csv file path safely using pathlib
        csv_file_path = patient_folder / TARGET_FILE_NAME

        # If the target file exists, load and process required columns
        if csv_file_path.is_file():
            df_vertebrae = pd.read_csv(csv_file_path)

            # Extract only the relevant column pair
            df_subset = df_vertebrae[['vertebra_name', 'n_lesions']]
            all_patient_records.append(df_subset)

# Combine all isolated dataframes into one master dataset
full_dataset = pd.concat(all_patient_records, ignore_index=True)

# ==================================================================
# 3. ANATOMICAL SORTING: Sorting Vertebrae Names Logically
# ==================================================================
def get_anatomical_key(vertebra_string):
    """
    Helper function to sort vertebrae anatomically (Cervical -> Thoracic -> Lumbar)
    instead of default alphabetical order (which incorrectly places L before T).
    """
    if pd.isna(vertebra_string):
        return (4, 0)

    # Standardize to uppercase for reliable matching
    name = str(vertebra_string).upper().strip()

    # Extract numerical index from the string (e.g., 'T10' -> 10)
    numbers = [int(s) for s in name if s.isdigit()]
    index_num = numbers[0] if numbers else 0

    # Group by spinal region order prefix
    if name.startswith('C'):   # Cervical
        return (1, index_num)
    elif name.startswith('T'): # Thoracic
        return (2, index_num)
    elif name.startswith('L'): # Lumbar
        return (3, index_num)
    else:                      # Fallback group
        return (4, index_num)

# Establish unique labels and sort them chronologically by spine structure
unique_vertebrae = full_dataset['vertebra_name'].dropna().unique()
anatomical_order = sorted(unique_vertebrae, key=get_anatomical_key)

# ==================================================================
# 4. VISUALIZATION: Generating Enhanced Structural Boxplots
# ==================================================================
sns.set_theme(style="darkgrid")
plt.figure(figsize=(14, 6))

# Render boxplot following the logical spinal column order
sns.boxplot(
    data=full_dataset,
    x='vertebra_name',
    y='n_lesions',
    order=anatomical_order,
    palette="flare",
    showfliers=False
)

# Overlay data points to observe lesion count distribution per bone segment
sns.stripplot(
    data=full_dataset,
    x='vertebra_name',
    y='n_lesions',
    order=anatomical_order,
    color="black",
    alpha=0.25,
    jitter=0.2
)

# Formatting chart labels and titles entirely in English
plt.title('Distribution of Spine Lesions Through Individual Vertebrae', fontsize=14, pad=15)
plt.xlabel('Vertebra Name (Anatomical Order)', fontsize=11)
plt.ylabel('Number of Detected Lesions', fontsize=11)
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.show()

In [ ]:
# 1. Base configuration and directory definitions
MAIN_FOLDER = Path(r"E:\DATA_Myelomy")
VERTEBRA_FILE = "BMD_radiomics_vertebrae_features.csv"
LESION_FILE = "BMD_radiomics_lesions_features.csv"

vertebra_records = []
lesion_records = []

# ==================================================================
# 2. DATA EXTRACTION & AGGREGATION PER PATIENT
# ==================================================================
if MAIN_FOLDER.exists():
    for patient_folder in MAIN_FOLDER.iterdir():
        if not patient_folder.is_dir():
            continue

        patient_id = patient_folder.name
        v_csv = patient_folder / VERTEBRA_FILE
        l_csv = patient_folder / LESION_FILE

        # Load and aggregate total vertebra volume per patient-vertebra segment
        if v_csv.is_file():
            df_v = pd.read_csv(v_csv)
            # Grouping prevents many-to-many exploding rows during later merges
            df_v_grouped = df_v.groupby('vertebra_name', as_index=False)['original_shape_VoxelVolume'].sum()
            df_v_grouped.rename(columns={'original_shape_VoxelVolume': 'vertebra_volume'}, inplace=True)
            df_v_grouped['patient_id'] = patient_id
            vertebra_records.append(df_v_grouped)

        # Load and aggregate total lesion load volume per patient-vertebra segment
        if l_csv.is_file():
            df_l = pd.read_csv(l_csv)
            df_l_grouped = df_l.groupby('vertebra_name', as_index=False)['original_shape_VoxelVolume'].sum()
            df_l_grouped.rename(columns={'original_shape_VoxelVolume': 'lesions_volume'}, inplace=True)
            df_l_grouped['patient_id'] = patient_id
            lesion_records.append(df_l_grouped)

# Combine records into clean consolidated datasets
full_vertebra_df = pd.concat(vertebra_records, ignore_index=True)
full_lesion_df = pd.concat(lesion_records, ignore_index=True)

# Merge datasets using a combined key: patient_id + vertebra_name
merged_df = pd.merge(
    full_vertebra_df,
    full_lesion_df,
    on=['patient_id', 'vertebra_name'],
    how='left'
)

# Fill missing lesion volumes with 0 (meaning zero lesions were found in that vertebra)
merged_df['lesions_volume'] = merged_df['lesions_volume'].fillna(0)

# ==================================================================
# 3. MATHEMATICAL COMPUTATIONS & FILTERING
# ==================================================================
# Formula: Lesion Ratio (%) = Lesion Volume / Total Bone Volume * 100
merged_df['lesion_percent'] = (
    merged_df['lesions_volume'] /
    (merged_df['vertebra_volume'] + merged_df['lesions_volume']) * 100
)

# Exclude S1 vertebra and any potential invalid rows
final_df = merged_df[merged_df['vertebra_name'] != 'S1'].copy()

# ==================================================================
# 4. ANATOMICAL SORTING
# ==================================================================
def get_anatomical_key(vertebra_string):
    """Sorts spine segments realistically from C1 down to L5."""
    if pd.isna(vertebra_string):
        return (4, 0)
    name = str(vertebra_string).upper().strip()
    digits = [int(s) for s in name if s.isdigit()]
    index_num = digits[0] if digits else 0

    if name.startswith('C'):
        return (1, index_num)
    elif name.startswith('T'):
        return (2, index_num)
    elif name.startswith('L'):
        return (3, index_num)
    else:
        return (4, index_num)

unique_vertebrae = final_df['vertebra_name'].dropna().unique()
anatomical_order = sorted(unique_vertebrae, key=get_anatomical_key)

# ==================================================================
# 5. VISUALIZATION
# ==================================================================
sns.set_theme(style="darkgrid")
plt.figure(figsize=(14, 6))

# Render boxplot following the anatomical path
sns.boxplot(
    data=final_df,
    x='vertebra_name',
    y='lesion_percent',
    order=anatomical_order,
    palette="viridis",
    showfliers=False
)

# Overlay individual data points to track patient variation
sns.stripplot(
    data=final_df,
    x='vertebra_name',
    y='lesion_percent',
    order=anatomical_order,
    color="black",
    alpha=0.3,
    jitter=0.2
)

# Set labels and layout settings in English
plt.title('Lesion Volume Ratio Distribution Across Spine Vertebrae (Excluding S1)', fontsize=13, pad=15)
plt.xlabel('Vertebra Name (Anatomical Spine Sequence)', fontsize=11)
plt.ylabel('Lesion Volume Ratio [%]', fontsize=11)
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.show()

In [ ]:
# 1. Global configurations
pd.set_option('future.no_silent_downcasting', True)
sns.set_theme(style="darkgrid")

# Define file paths and target parameters
DATA_DIRECTORY = r"E:\DATA_Myelomy"
CLINICAL_FILE_PATH = r"E:\Clinical_data\Table_clinical_data.csv"
TARGET_IMAGE_TYPE = "CaSupp_25"
GROUP_COLUMN = "ISS classification"

# ==================================================================
# 2. DATA LOADING & PATIENT AGGREGATION
# ==================================================================
# Load all datasets dynamically using your custom directory scanner function
all_datasets = merged_lesions_csv(DATA_DIRECTORY)

# Isolate lesion and vertebrae dataframes for the requested image modality
df_lesions_raw = all_datasets.get(TARGET_IMAGE_TYPE)
# Note: Ensure your helper covers vertebrae or adjust key names if they share keys
df_vertebrae_raw = all_datasets.get(TARGET_IMAGE_TYPE)

# Group by patient to sum up total volumes (prevents many-to-many merge explosion)
df_lesions_agg = (
    df_lesions_raw.groupby('patient', as_index=False)['original_shape_VoxelVolume']
    .sum()
    .rename(columns={'original_shape_VoxelVolume': 'lesions_volume'})
)

df_vertebrae_agg = (
    df_vertebrae_raw.groupby('patient', as_index=False)['original_shape_VoxelVolume']
    .sum()
    .rename(columns={'original_shape_VoxelVolume': 'vertebrae_volume'})
)

# Combine consolidated volume columns
volume_data = pd.merge(df_lesions_agg, df_vertebrae_agg, on='patient', how='inner')

# ==================================================================
# 3. CLINICAL DATA MERGE & CALCULATIONS
# ==================================================================
clinical_df = pd.read_csv(CLINICAL_FILE_PATH, encoding="cp1252")

# Combine the aggregated volumes with staging classifications
df_final = pd.merge(
    volume_data,
    clinical_df[['Patient ID', GROUP_COLUMN]],
    left_on='patient',
    right_on='Patient ID'
)

# Compute the total lesion volume burden index ratio per patient
df_final['volume_ratio'] = (
    df_final['lesions_volume'] /
    (df_final['vertebrae_volume'] + df_final['lesions_volume']) * 100
)

# Clean missing target values
df_final = df_final.dropna(subset=[GROUP_COLUMN, 'volume_ratio']).reset_index(drop=True)

# ==================================================================
# 4. STATISTICAL TESTING: Kruskal-Wallis Test
# ==================================================================
# Build analytical groups dynamically based on staging strings
groups = [
    group_data['volume_ratio'].values
    for name, group_data in df_final.groupby(GROUP_COLUMN)
]

print("--- Kruskal-Wallis Test Results ---")
if len(groups) > 1 and all(len(g) > 0 for g in groups):
    stat, p_value = kruskal(*groups)
    print(f"H-statistic: {stat:.4f}")
    print(f"p-value:     {p_value:.4e}")
    print(f"Significant: {p_value < 0.05}")
else:
    stat, p_value = np.nan, np.nan
    print("Error: Insufficient staging cohorts available to compute metrics.")
print("-" * 40)

# ==================================================================
# 5. VISUALIZATION
# ==================================================================
plt.figure(figsize=(8, 6))

# Chronological sorting for stages on the X-axis (Stage 1 -> Stage 2 -> Stage 3)
stage_order = sorted(df_final[GROUP_COLUMN].unique())

# Render boxplot layout
sns.boxplot(
    data=df_final,
    x=GROUP_COLUMN,
    y="volume_ratio",
    order=stage_order,
    palette="muted",
    showfliers=False
)

# Overlay individual patient metrics for clear sample density profile
sns.stripplot(
    data=df_final,
    x=GROUP_COLUMN,
    y="volume_ratio",
    order=stage_order,
    color="black",
    alpha=0.35,
    jitter=0.2
)

# English labelling configurations
plt.xlabel("ISS Clinical Classification", fontsize=11)
plt.ylabel("Total Lesions Volume Ratio [%]", fontsize=11)

if not np.isnan(p_value):
    plt.title(
        f"Distribution of Total Lesions Volume Ratio by ISS Stage ({TARGET_IMAGE_TYPE})\n"
        f"Kruskal-Wallis H = {stat:.2f} (p = {p_value:.4e})",
        fontsize=12,
        pad=15
    )
else:
    plt.title(f"Distribution of Total Lesions Volume Ratio by ISS Stage ({TARGET_IMAGE_TYPE})", fontsize=12, pad=15)

plt.tight_layout()
plt.show()

In [ ]:
# ==================================================================
# USER CONFIGURATION: Choose your clinical grouping column here
# ==================================================================
# Change this string to analyze different clinical variables
TARGET_CLINICAL_COLUMN = "ISS classification"  # e.g., "ISS classification", "M-protein type", "Light chain type"

print(f"Analyzing lesion volume ratios grouped by: '{TARGET_CLINICAL_COLUMN}'")
print("-" * 70)

# ==================================================================
# 1. PATH CONFIGURATIONS
# ==================================================================
pd.set_option('future.no_silent_downcasting', True)
sns.set_theme(style="darkgrid")

main_folder = r"E:\DATA_Myelomy"
clinical_file_path = r"E:\Clinical_data\Table_clinical_data.csv"

lesion_records = []
vertebra_records = []

# ==================================================================
# 2. DATA LOADING & AGGREGATION
# ==================================================================
for folder_name in os.listdir(main_folder):
    folder_path = os.path.join(main_folder, folder_name)

    lesion_csv = os.path.join(folder_path, "monoe_40kev_radiomics_lesions_features.csv")
    vertebra_csv = os.path.join(folder_path, "monoe_40kev_radiomics_vertebrae_features.csv")

    if os.path.isfile(lesion_csv):
        df_l = pd.read_csv(lesion_csv).filter(items=["vertebra_name", "original_shape_VoxelVolume"])
        df_l["patient"] = folder_name
        lesion_records.append(df_l)

    if os.path.isfile(vertebra_csv):
        df_v = pd.read_csv(vertebra_csv).filter(items=["vertebra_name", "original_shape_VoxelVolume"])
        df_v["patient"] = folder_name
        vertebra_records.append(df_v)

# Safeguard against completely missing datasets
if not lesion_records or not vertebra_records:
    raise FileNotFoundError("Could not find matching radiomics CSV files. Check your main folder path.")

df_lesions_raw = pd.concat(lesion_records, ignore_index=True)
df_vertebra_raw = pd.concat(vertebra_records, ignore_index=True)

# Group rows to calculate accurate totals per bone element per patient
df_lesions_grouped = (
    df_lesions_raw.groupby(["patient", "vertebra_name"], as_index=False)["original_shape_VoxelVolume"]
    .sum()
    .rename(columns={"original_shape_VoxelVolume": "lesions_volume"})
)

df_vertebra_grouped = (
    df_vertebra_raw.groupby(["patient", "vertebra_name"], as_index=False)["original_shape_VoxelVolume"]
    .sum()
    .rename(columns={"original_shape_VoxelVolume": "vertebra_volume"})
)

# Merge datasets
df = pd.merge(df_lesions_grouped, df_vertebra_grouped, on=["patient", "vertebra_name"], how="inner")
df = df[df["vertebra_name"] != "S1"].copy()

# ==================================================================
# 3. SPINAL REGION STRUCTURAL EXTENSION
# ==================================================================
df['region'] = df['vertebra_name'].str[0].map({
    'C': 'Cervical',
    'T': 'Thoracic',
    'L': 'Lumbar'
})

# ==================================================================
# 4. DYNAMIC CLINICAL MERGE & BURDEN ANALYSIS
# ==================================================================
clinical_df = pd.read_csv(clinical_file_path, encoding="cp1252")

# Safety validation: Make sure the chosen clinical column actually exists
if TARGET_CLINICAL_COLUMN not in clinical_df.columns:
    raise KeyError(f"The column '{TARGET_CLINICAL_COLUMN}' was not found in your clinical CSV. "
                   f"Available options are: {list(clinical_df.columns)}")

# Merge image stats with the dynamically selected clinical target column
df_final = pd.merge(
    df,
    clinical_df[['Patient ID', TARGET_CLINICAL_COLUMN]],
    left_on='patient',
    right_on='Patient ID'
)

# Compute burden metrics
df_final['volume_ratio'] = (
    df_final['lesions_volume'] /
    (df_final['vertebra_volume'] + df_final['lesions_volume']) * 100
)

# Drop any rows containing missing strings in our user-selected grouping target column
df_final = df_final.dropna(subset=[TARGET_CLINICAL_COLUMN, 'volume_ratio']).reset_index(drop=True)

# ==================================================================
# 5. DYNAMIC MULTI-VARIABLE VISUALIZATION
# ==================================================================
if not df_final.empty:
    plt.figure(figsize=(12, 6))

    # Dynamically extract and sort unique clinical classes for uniform X-axis rendering
    group_order = sorted(df_final[TARGET_CLINICAL_COLUMN].unique())

    # Build the grouped boxplot using your custom configuration choices
    sns.boxplot(
        data=df_final,
        x=TARGET_CLINICAL_COLUMN,
        y="volume_ratio",
        hue="region",
        order=group_order,
        hue_order=["Cervical", "Thoracic", "Lumbar"],
        palette="muted",
        showfliers=False
    )

    # Dynamic labelling that alters strings based on your variables automatically
    plt.xlabel(TARGET_CLINICAL_COLUMN, fontsize=11)
    plt.ylabel("Lesions Volume Ratio [%]", fontsize=11)
    plt.title(f"Lesions Volume Ratio by {TARGET_CLINICAL_COLUMN} and Spinal Region (monoe_40kev)", fontsize=13, pad=15)
    plt.legend(title="Spinal Region")

    # Adjust spacing dynamically if category strings run wide
    plt.xticks(rotation=15, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print("Warning: The resulting filtered DataFrame is empty. Check data pairings.")